In [1]:
######################################################################################################
# ALUNO : Daniel Marques Diniz Maia
# MATRÍCULA : 2026061836
# TURMA : TM
######################################################################################################

In [2]:
# Definindo as funções de busca

def inverse_parabolic_minimize(f, df, x0, x1, x2, tol=1e-10, max_iter=100):
    """
    Function minimization using inverse parabolic interpolation
    applied to the derivative f'(x).

    We seek a root of f'(x) = 0.

    Parameters
    ----------
    f : callable
        Function to minimize.
    df : callable
        First derivative of f.
    x0, x1, x2 : float
        Three initial estimates.
    tol : float
        Convergence tolerance.
    max_iter : int
        Maximum number of iterations.

    Returns
    -------
    xmin : float
        Estimated location of the minimum.
    fmin : float
        Function value at the minimum.
    """

    for iteration in range(1, max_iter + 1):

        # Derivative values
        d0 = df(x0)
        d1 = df(x1)
        d2 = df(x2)

        # Exact stationary point
        if abs(d0) < tol:
            return x0, f(x0)

        if abs(d1) < tol:
            return x1, f(x1)

        if abs(d2) < tol:
            return x2, f(x2)

        # Check for repeated derivative values
        if (d0 == d1) or (d0 == d2) or (d1 == d2):
            raise ValueError(
                "Derivative values must be distinct."
            )

        # Inverse parabolic interpolation
        # x = x(d), evaluated at d = 0
        x3 = (
            x0 * d1 * d2 / ((d0 - d1) * (d0 - d2))
            + x1 * d0 * d2 / ((d1 - d0) * (d1 - d2))
            + x2 * d0 * d1 / ((d2 - d0) * (d2 - d1))
        )

        # Convergence criterion
        if abs(x3 - x2) < tol:
            return x3, f(x3)

        # Update points
        x0 = x1
        x1 = x2
        x2 = x3

    raise RuntimeError(
        "Maximum number of iterations reached.")
    
import numpy as np
import sympy as sp
from scipy.optimize import minimize

def minimos(funcao, derfuncao, chute):
    '''
    Função Python que recebe uma função vetorial funcao(v) e sua derivada (ou gradiente) e um chute inicial para a busca (TYPE: np.array) e imprime tudo que precisamos
    '''
    #SIMPLEX
    res1 = minimize(funcao, chute, method='nelder-mead', options={'xatol': 1e-8})
    print(f'SIMPLEX - xmin = {[f'{a:.5e}' for a in res1.x]} /// f(xmin) = {res1.fun:.5e}')

    # POWELL
    res2 = minimize(funcao, chute, method='Powell', jac=None)
    print(f'POWELL - xmin = {[f'{a:.5e}' for a in res2.x]} /// f(xmin) = {res2.fun:.5e}')

    # BFGS
    res3 = minimize(funcao, chute, method='BFGS', jac=derfuncao)
    print(f'BFGS - xmin = {[f'{a:.5e}' for a in res3.x]} /// f(xmin) = {res3.fun:.5e}')






In [3]:
s = sp.symbols('s')
t = sp.symbols('t')

# ------------------------------------------------------------------------------------------------------------------
# Encontrando o mínimo para Rosenbrock

print('='*60)
print('ROSENBROCK\n')

def rosen(x):
    """The Rosenbrock function"""
    return sum(100.0*(x[1:]-x[:-1]**2.0)**2.0 + (1-x[:-1])**2.0)

def rosen_der(x):
    xm = x[1:-1]
    xm_m1 = x[:-2]
    xm_p1 = x[2:]
    der = np.zeros_like(x)
    der[1:-1] = 200*(xm-xm_m1**2) - 400*(xm_p1 - xm**2)*xm - 2*(1-xm)
    der[0] = -400*x[0]*(x[1]-x[0]**2) - 2*(1-x[0])
    der[-1] = 200*(x[-1]-x[-2]**2)
    return der

minimos(rosen,rosen_der,[0.5,0.5])


# ------------------------------------------------------------------------------------------------------------------
# Encontrando o mínimo para uma Lorentziana 3d (lorentziana(x)*lorentziana(y))

print('='*60)
print('LORENTZIANA\n')

# Derivando para o BFGS
sym_lorentziana = -(1.5/((s)**2+1.5**2))*(1.5/((t)**2+1.5**2)) 

dfds = sp.lambdify((s, t), sp.diff(sym_lorentziana, s), "numpy")
dfdt = sp.lambdify((s, t), sp.diff(sym_lorentziana, t), "numpy")

def grad_lorentziana(v):
    return np.array([dfds(*v), dfdt(*v)], dtype=float)

# Função para o minimize (deve ser vetorial, e obter a função numérica facilita a conversão)

lorentziana_num = sp.lambdify((s, t), sym_lorentziana, "numpy")
lorentziana_vec = lambda v: lorentziana_num(*v)

# Obtendo os resultados
minimos(lorentziana_vec, grad_lorentziana,[0.5, 0.5])



# Para os exercícios do slide, podemos não utilizar o valor das constantes, uma vez que elas não alteram para qual valor da variável, o valor da função é mínimo, mas sim
# o valor mínimo da função (xmin para f(x) é o mesmo para C*f(x), mas f(xmin)!=C*f(xmin)). Então, por conveniência, substituiremos todas as constantes por 1

# ------------------------------------------------------------------------------------------------------------------
# Potencial de Lennard-Jones

print('='*60)
print('Potencial de Lennard-Jones\n')

def V(r):
    return 4 * (r**-12 - r**-6)
def dV(r):
    return -48 * r**-13 + 24 * r**-7

V_vec = lambda v: V(v[0])
dV_vec = lambda v: np.array([dV(v[0])])

minimos(V_vec,dV_vec, [1])
# A função é 1D, então inversão parabólica funciona (precisa de funções escalares)
xmin, fmin = inverse_parabolic_minimize(V, dV, x0=0.1,  x1=1.1, x2=1.2)
print(f'INVERSÃO PARABÓLICA - xmin = {xmin:.5e} /// f(xmin) = {fmin:.5e}')

# ------------------------------------------------------------------------------------------------------------------
# Função de onda do hidrogênio

print('='*60)
print('Função de onda do hidrogênio\n')

def psi(s):
    return (27 - 18*s + 2*s**2) * np.exp(-s/3)
def dpsi(s):
    return (-27 + 10*s - (2/3)*s**2) * np.exp(-s/3)

psi_vec = lambda v: psi(v[0])
dpsi_vec = lambda v: np.array([dpsi(v[0])])

minimos(psi_vec, dpsi_vec, [3.0])

# Função 1D, então a inversão parabólica funciona (precisa de funções escalares)
xmin, fmin = inverse_parabolic_minimize(psi, dpsi, x0=3.0, x1=3.5, x2=4.0)
print(f'INVERSÃO PARABÓLICA - xmin = {xmin:.5e} /// f(xmin) = {fmin:.5e}')

ROSENBROCK

SIMPLEX - xmin = ['1.00000e+00', '1.00000e+00'] /// f(xmin) = 2.62598e-18
POWELL - xmin = ['1.00000e+00', '1.00000e+00'] /// f(xmin) = 4.12452e-27
BFGS - xmin = ['1.00000e+00', '1.00000e+00'] /// f(xmin) = 1.49543e-15
LORENTZIANA

SIMPLEX - xmin = ['-6.81946e-09', '-7.42064e-09'] /// f(xmin) = -4.44444e-01
POWELL - xmin = ['-9.34275e-09', '-9.34275e-09'] /// f(xmin) = -4.44444e-01
BFGS - xmin = ['6.76088e-06', '6.76088e-06'] /// f(xmin) = -4.44444e-01
Potencial de Lennard-Jones

SIMPLEX - xmin = ['1.12246e+00'] /// f(xmin) = -1.00000e+00
POWELL - xmin = ['1.12246e+00'] /// f(xmin) = -1.00000e+00
BFGS - xmin = ['1.12246e+00'] /// f(xmin) = -1.00000e+00
INVERSÃO PARABÓLICA - xmin = 1.12246e+00 /// f(xmin) = -1.00000e+00
Função de onda do hidrogênio

SIMPLEX - xmin = ['3.53137e+00'] /// f(xmin) = -3.58195e+00
POWELL - xmin = ['3.53137e+00'] /// f(xmin) = -3.58195e+00
BFGS - xmin = ['3.53137e+00'] /// f(xmin) = -3.58195e+00
INVERSÃO PARABÓLICA - xmin = 3.53137e+00 /// f(xmin) =